# Who the Models Work For

**Computational Text Analysis Pack · Module 26 — Fine-Tuning Pretrained Models**

*Technique:* Per-language performance against data volume; measuring the inequity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/13-african-nlp-models/26b_low_resource_reality.ipynb)

Data: `masakhanews.csv`, `masakhaner.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'masakhaner.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_ner   = pd.read_csv('masakhaner.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Aggregate scores hide the thing that matters most for African DH: **performance
is not evenly distributed across languages.** A model reported at 0.85 may be at
0.93 on Swahili and 0.61 on Wolof, and if your project is on Wolof the headline
number told you nothing.

This notebook measures the spread and relates it to the one variable that
predicts it.

## The tooling

**HuggingFace `transformers`** provides the models and the training loop
(`Trainer`), **`datasets`** the data plumbing, and **PyTorch** the engine. For
serious runs you would reach for a GPU; everything below is sized to run on CPU
so the workflow is learnable without one.

The alternative to fine-tuning is **frozen features**: run the pretrained model
once, keep the vectors, and train a cheap classifier on top. It is far faster,
often nearly as good, and is what module 23d already did with LaBSE.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

bank = np.load("labse_headlines.npy")
index = pd.read_csv("labse_headlines_index.csv")
ok = index["category"].notna().values
X, meta = bank[ok], index.loc[ok].reset_index(drop=True)

tr, te = train_test_split(np.arange(len(X)), test_size=0.35, random_state=0,
                          stratify=meta["category"])
clf = LogisticRegression(max_iter=2000).fit(X[tr], meta["category"].iloc[tr])
pred = clf.predict(X[te])

rows = []
langs = meta["language"].iloc[te].values
gold = meta["category"].iloc[te].values
for lang in sorted(set(langs)):
    m = langs == lang
    if m.sum() < 8:
        continue
    rows.append({"language": lang, "n_test": int(m.sum()),
                 "accuracy": accuracy_score(gold[m], pred[m])})
per = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
print(per.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print(f"overall accuracy      : {accuracy_score(gold, pred):.3f}")
print(f"best language         : {per.iloc[0]['language']} {per.iloc[0]['accuracy']:.3f}")
print(f"worst language        : {per.iloc[-1]['language']} {per.iloc[-1]['accuracy']:.3f}")
print(f"spread                : {per['accuracy'].max() - per['accuracy'].min():.3f}")
print()
print("The overall figure is an average over that spread. Nobody's project is")
print("conducted on the average.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(per["language"][::-1], per["accuracy"][::-1], color="steelblue")
ax.axvline(accuracy_score(gold, pred), ls="--", color="crimson",
           label="overall average")
ax.set_xlabel("accuracy"); ax.legend()
ax.set_title("The headline number hides this")
plt.tight_layout()

## What predicts a language's score

Not anything intrinsic to the language. The predictor is how much of it existed,
digitised, when the model was trained — which is a function of colonial language
policy, missionary printing, state investment and commercial value.

In [ ]:
# A rough proxy: corpus volume per language, and the NER benchmark's coverage.
vol = df.groupby("language")["n_words"].sum().rename("corpus_words")
ner_cov = df_ner.groupby("language").size().rename("ner_sentences")
merged = per.set_index("language").join(vol).join(ner_cov)
merged["ner_sentences"] = merged["ner_sentences"].fillna(0)
print(merged.to_string(float_format=lambda v: f"{v:,.3f}"))

sub = merged.dropna(subset=["corpus_words"])
if len(sub) > 3:
    r = np.corrcoef(np.log(sub["corpus_words"]), sub["accuracy"])[0, 1]
    print(f"\ncorrelation of accuracy with log corpus size: {r:.3f}")
    print()
    print("Our corpus size is a weak proxy for pretraining volume, so read this")
    print("as indicative. The published MasakhaNER papers show the relationship")
    print("clearly: languages with more pretraining text score higher, and the")
    print("gap does not close by trying harder at inference time.")

## What to do about it

Not "use a bigger model" — that mostly widens the gap, because scaling is fed by
the same skewed data.

- **Report per language, always.** An aggregate score across languages is close
  to meaningless for a reader who works in one of them.
- **Set the baseline in the same language.** Compare against a classical method
  on *your* language, not against the model's English score.
- **Consider contributing data.** MasakhaNER and MasakhaNEWS exist because
  researchers annotated them. That is a DH contribution as real as a paper.
- **Be willing to conclude the tool does not work here.** Modules 2–19 are not a
  fallback; for many languages they are the better method.

### 🔧 Your turn 1

Pick the two lowest-scoring languages and run a classical TF-IDF classifier on just those. Does the classical method beat the pretrained one there?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Often it does, and that reversal is the most useful finding in this module. A
pretrained model's advantage comes from transfer, and where there was little to
transfer from, a method that learns everything from your labelled data is
competitive or better. The general rule: the less a model knows about your
language, the more the classical toolkit is worth — which is the opposite of
how these tools are usually marketed.

</details>